In [ ]:
!pip install tensorflow_addons

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.0/591.0 KB 9.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
import string
import re
import unicodedata
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

class MTM:
  def __init__(self, name, batch_size):
    self.batch_size = batch_size
    self.name = name

  def unicode_to_ascii(self, s):
    ''' unicode string '''
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

  def preprocess_sentence(self, s):
    ''' 
        unicode sentence to ascii 
        keep any character and .?!,¿ punctiuations
        add space between punctiuation and words for decoding
        adding [SOS] and [EOS] special tokens
    '''
    s = self.unicode_to_ascii(s.lower().strip())
    
    s = re.sub(r'[^ a-z.?!,¿]', '', s)
    s = re.sub(r"([?.!,¿])", r" \1 ", s)
    s = re.sub(r'[" "]+', " ", s)
    
    return '[SOS] ' + s + ' [EOS]'

  def tokenize(self, data):
    tok = Tokenizer(oov_token='[UNK]', filters='', lower=False)
    tok.fit_on_texts(data)
    sequenced_data = tok.texts_to_sequences(data)

    return sequenced_data, tok

  def create_dataset(self, texts, in_targets, ot_targets, batch_size):

    dataset = tf.data.Dataset.from_tensor_slices(((texts, in_targets), ot_targets))
    dataset = dataset.cache()
    dataset = dataset.shuffle(len(texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(batch_size)

    return dataset

  def read_data(self, path):
    inputs = []
    targets = []
    with open(path) as f:
      for line in f.readlines():
        line = line.split('\t')
        inputs.append(self.preprocess_sentence(line[1]))
        targets.append(self.preprocess_sentence(line[0]))
    
    return inputs, targets

  def call(self, path):
    inputs, targets = self.read_data(path)
    print(len(inputs))
    inputs_seq_data, input_tok = self.tokenize(inputs)
    targets_seq_data, target_tok = self.tokenize(targets)

    x_train, x_val, y_train, y_val = train_test_split(inputs_seq_data, targets_seq_data)

    train_sp = pad_sequences(x_train, padding='post')
    train_en_inputs = pad_sequences([seq[:-1] for seq in y_train], padding='post')
    train_en_outputs = pad_sequences([seq[1:] for seq in y_train], padding='post')

    val_sp = pad_sequences(x_val, padding='post')
    val_en_inputs = pad_sequences([seq[:-1] for seq in y_val], padding='post')
    val_en_outputs = pad_sequences([seq[1:] for seq in y_val], padding='post')

    train_dataset = self.create_dataset(train_sp, train_en_inputs, train_en_outputs, self.batch_size)
    val_dataset = self.create_dataset(val_sp, val_en_inputs, val_en_outputs, self.batch_size)

    return train_dataset, val_dataset, input_tok, target_tok


In [ ]:
##
BATCH_SIZE = 64
##

mtm = MTM('spa-en', BATCH_SIZE)
train_dataset, val_dataset, input_tok, target_tok = mtm.call('/content/drive/MyDrive/NTM/spa.txt')

139705


In [ ]:
for (sp, en_inputs), en_labels in train_dataset.take(1):
  break
# print(sp.shape)
print(en_inputs[0])
print(en_labels[0])

tf.Tensor(
[  2  19   8 579  21  11   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0], shape=(81,), dtype=int32)
tf.Tensor(
[ 19   8 579  21  11   3   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0], shape=(81,), dtype=int32)


# Transformer

In [ ]:
import tensorflow as ts
from tensorflow import keras
from keras.layers import Layer, MultiHeadAttention, Dense, Add, LayerNormalization, Embedding, Dropout

In [ ]:
class PositionalEmbedding(Layer):
  def __init__(self, vocab_size, em_units):
    super().__init__()
    self.em_units = em_units
    self.embedding = Embedding(vocab_size, em_units, mask_zero=True) 
    self.pos_encoding = self.positional_encoding(length=2048, depth=em_units)

  def positional_encoding(self, length, depth):
    depth = depth/2

    positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
    depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

    angle_rates = 1 / (10000**depths)         # (1, depth)
    angle_rads = positions * angle_rates      # (pos, depth)

    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1) 

    return tf.cast(pos_encoding, dtype=tf.float32)
    
  def compute_mask(self, *args, **kwargs):
    return self.embedding.compute_mask(*args, **kwargs)

  def call(self, x):
    length = tf.shape(x)[1]
    x = self.embedding(x)
    x *= tf.math.sqrt(tf.cast(self.em_units, tf.float32))
    x = x + self.pos_encoding[tf.newaxis, :length, :]

    return x

In [ ]:
embed_sp = PositionalEmbedding(vocab_size=len(input_tok.word_index)+1, em_units=512)
embed_en = PositionalEmbedding(vocab_size=len(target_tok.word_index)+1, em_units=512)

sp_emb = embed_sp(sp)
en_emb = embed_en(en_inputs)

In [ ]:
class BaseAttention(Layer):
  def __init__(self, **kwargs):
    super(BaseAttention, self).__init__()
    self.mha = MultiHeadAttention(**kwargs)
    self.nrm = LayerNormalization()
    self.add = Add()

In [ ]:
class CrossAttention(BaseAttention):
  def call(self, x, context):
    attn_output, attn_scores = self.mha(
        query=x,
        key=context,
        value=context,
        return_attention_scores=True)
    self.last_attn_scores = attn_scores
    x = self.add([attn_output, x])
    x = self.nrm(x)

    return x

In [ ]:
ca = CrossAttention(num_heads=2, key_dim=512)
print(sp_emb.shape)
print(en_emb.shape)
print(ca(en_emb, sp_emb).shape)

(64, 80, 512)
(64, 81, 512)
(64, 81, 512)


In [ ]:
class SelfAttention(BaseAttention):
  def call(self, x):
    atten_output = self.mha(
        query=x,
        key=x,
        value=x
    )
    x = self.add([x, atten_output])
    x = self.nrm(x)

    return x

In [ ]:
sa = SelfAttention(num_heads=2, key_dim=512)
print(sa(sp_emb).shape)

(64, 80, 512)


In [ ]:
class MaskedAttention(BaseAttention):
  def call(self, x):
    atten_output = self.mha(
        query=x,
        key=x,
        value=x,
        use_causal_mask = True
    )
    x = self.add([x, atten_output])
    x = self.nrm(x)

    return x

In [ ]:
ma = MaskedAttention(num_heads=2, key_dim=512)
print(ma(en_emb).shape)

(64, 81, 512)


In [ ]:
class FeedForwordLayer(Layer):
  def __init__(self, ffd_units, em_units, drop_out=.1):
    super(FeedForwordLayer, self).__init__()
    self.ffd = keras.models.Sequential([
        Dense(ffd_units, activation='relu'),
        Dense(em_units, activation='relu'),
        Dropout(drop_out)
    ])

    self.add = Add()
    self.nrm = LayerNormalization()

  def call(self, x):
    output = self.ffd(x)
    x = self.add([x, output])
    x = self.nrm(x)

    return x

In [ ]:
ffd = FeedForwordLayer(128, 512)
ffd(sa(sp_emb)).shape

TensorShape([64, 80, 512])

In [ ]:
class EncoderLayer(Layer):
  def __init__(self, num_heads, dim, ffd_units, drop_out):
    super(EncoderLayer, self).__init__()
    self.atn = SelfAttention(num_heads=num_heads, key_dim=dim)
    self.ffd = FeedForwordLayer(ffd_units, dim, drop_out)

  def call(self, x):
    x = self.atn(x)
    x = self.ffd(x)

    return x

In [ ]:
class Encoder(Layer):
  def __init__(self, n_layers, vocab_size, dim, num_heads, ffd_units, drop_out):
    super(Encoder, self).__init__()
    self.n_layers = n_layers
    self.pos = PositionalEmbedding(vocab_size, dim)
    self.layers = [
        EncoderLayer(num_heads, dim, ffd_units, drop_out) for _ in range(n_layers)
    ]

  def call(self, x):
    x = self.pos(x)
    for i in range(self.n_layers):
      x = self.layers[i](x)

    return x

In [ ]:
sample_encoder = Encoder(4,
                         len(input_tok.word_index)+1,
                         100,
                         4,
                         512,
                         .2)

sample_encoder_output = sample_encoder(sp, training=False)

# Print the shape.
print(sp.shape)
print(sample_encoder_output.shape)

(64, 80)
(64, 80, 100)


In [ ]:
class DecoderLayer(Layer):
  def __init__(self, num_heads, dim, ffd_units, dropout):
    super(DecoderLayer, self).__init__()
    self.mal = MaskedAttention(num_heads=num_heads, key_dim=dim)
    self.cal = CrossAttention(num_heads=num_heads, key_dim=dim)
    self.ffl = FeedForwordLayer(ffd_units, dim, dropout)

  def call(self, x, context):
    x = self.mal(x)
    x = self.cal(x, context)
    x = self.ffl(x)

    return x

In [ ]:
class Decoder(Layer):
  def __init__(self, n_layers, vocab_size, dim, num_heads, ffd_units, dropout):
    super(Decoder, self).__init__()
    self.n_layers = n_layers
    self.pel = PositionalEmbedding(vocab_size, dim)
    self.dec = [DecoderLayer(num_heads, dim, ffd_units, dropout) for _ in range(n_layers)]

  def call(self, x, context):
    x = self.pel(x)
    for layer in self.dec:
      x = layer(x, context)

    return x

In [ ]:
class Transformer(keras.Model):
  def __init__(self, n_layers, input_vocab_size, target_vocab_size, dim, num_heads, ffd_units, drop_out):
    super(Transformer, self).__init__()
    self.encoder = Encoder(n_layers,
                           input_vocab_size,
                           dim,
                           num_heads,
                           ffd_units,
                           drop_out)
    self.decoder = Decoder(n_layers,
                           target_vocab_size,
                           dim,
                           num_heads,
                           ffd_units,
                           drop_out)
    self.out = Dense(target_vocab_size)

  def call(self, inputs):
    context, x = inputs
    context = self.encoder(context)
    x = self.decoder(x, context)
    logits = self.out(x)

    try:
      del logits._keras_mask
    except AttributeError:
      pass

    return logits

In [ ]:
##################################
n_layers = 6
input_vocab_size=len(input_tok.word_index)+1
target_vocab_size=len(target_tok.word_index)+1
dim = 300
num_heads = 6
ffd_units = 256
dropout = .3
##################################

model = Transformer(n_layers, input_vocab_size, target_vocab_size, dim, num_heads, ffd_units, dropout)
model((sp, en_inputs)).shape

TensorShape([64, 81, 14163])

In [ ]:
model.summary()

Model: "transformer"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 encoder_1 (Encoder)         multiple                  21928236  
                                                                 
 decoder (Decoder)           multiple                  31173036  
                                                                 
 dense_34 (Dense)            multiple                  4263063   
                                                                 
Total params: 57,364,335
Trainable params: 57,364,335
Non-trainable params: 0
_________________________________________________________________


In [ ]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
  def __init__(self, d_model, warmup_steps=4000):
    super().__init__()

    self.d_model = d_model
    self.d_model = tf.cast(self.d_model, tf.float32)

    self.warmup_steps = warmup_steps

  def __call__(self, step):
    step = tf.cast(step, dtype=tf.float32)
    arg1 = tf.math.rsqrt(step)
    arg2 = step * (self.warmup_steps ** -1.5)

    return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

  def get_config(self):
    config = {
        'd_model': self.d_model,
        'warmup_steps': self.warmup_steps
    }
    return config

In [ ]:
def masked_loss(label, pred):
  mask = label != 0
  loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')
  loss = loss_object(label, pred)

  mask = tf.cast(mask, dtype=loss.dtype)
  loss *= mask

  loss = tf.reduce_sum(loss)/tf.reduce_sum(mask)
  return loss


def masked_accuracy(label, pred):
  pred = tf.argmax(pred, axis=2)
  label = tf.cast(label, pred.dtype)
  match = label == pred

  mask = label != 0

  match = match & mask

  match = tf.cast(match, dtype=tf.float32)
  mask = tf.cast(mask, dtype=tf.float32)
  return tf.reduce_sum(match)/tf.reduce_sum(mask)

In [ ]:
learning_rate = CustomSchedule(dim)
optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)
model.compile(
    loss=masked_loss,
    optimizer=optimizer,
    metrics=[masked_accuracy])

In [ ]:
path = '/content/drive/MyDrive/NTM/checkpoints'
checkpoints = keras.callbacks.ModelCheckpoint(path, save_best_only=True)
earlystopping = keras.callbacks.EarlyStopping(patience=1)
tensorboard = keras.callbacks.TensorBoard('/content/drive/MyDrive/NTM/logs')

In [ ]:
input_vocab_size
target_vocab_size

14163

In [ ]:
model.fit(train_dataset,
          epochs=20, 
          validation_data=val_dataset,
          callbacks=[earlystopping])

Epoch 1/20
1638/1638 [==============================] - 1353s 791ms/step - loss: 4.2635 - masked_accuracy: 0.4259 - val_loss: 2.3531 - val_masked_accuracy: 0.6193
Epoch 2/20
1638/1638 [==============================] - 1289s 787ms/step - loss: 1.8918 - masked_accuracy: 0.6761 - val_loss: 1.7274 - val_masked_accuracy: 0.6958
Epoch 3/20
1638/1638 [==============================] - 1287s 786ms/step - loss: 1.5437 - masked_accuracy: 0.7140 - val_loss: 1.5123 - val_masked_accuracy: 0.7265
Epoch 4/20
1638/1638 [==============================] - 1286s 785ms/step - loss: 1.2173 - masked_accuracy: 0.7631 - val_loss: 1.3156 - val_masked_accuracy: 0.7610
Epoch 5/20
1638/1638 [==============================] - 1286s 785ms/step - loss: 1.0058 - masked_accuracy: 0.8003 - val_loss: 1.2423 - val_masked_accuracy: 0.7758
Epoch 6/20
1498/1638 [==========================>...] - ETA: 1:41 - loss: 0.8627 - masked_accuracy: 0.8267

In [ ]:
### END OF THE MODEL

In [ ]:
#@title seq2seq with rnn
variable_name = ""
import tensorflow as tf
from tensorflow import keras
from keras.layers import Layer, Embedding, Bidirectional, LSTM, Dense, Input, GRU, LSTMCell

#encoder architecture
class Encoder(keras.Model):
  def __init__(self, vocab_size, embedding_dim, enc_units, batch_size):
    super(Encoder, self).__init__()
    self.batch_size = batch_size
    self.enc_units = enc_units
    self.embedding = Embedding(vocab_size, embedding_dim)
    self.bi_lstm = Bidirectional(GRU(enc_units, return_sequences=True, return_state=True))

  def call(self, x, hidden):
    x = self.embedding(x)
    output, forward, backward = self.bi_lstm(x, initial_state = hidden)
    return output, tf.concat([forward, backward], axis=-1)
  
  def initialize_hidden_state(self):
    # Initialize hidden state with zeros of shape [batch_size, enc_units]
    return [tf.zeros((self.batch_size, self.enc_units)), tf.zeros((self.batch_size, self.enc_units))]

In [ ]:
#@title Default title text
vocab_size = len(input_tok.word_index)+1
input_dataset, target_dataset = next(iter(dataset))

encoder = Encoder(vocab_size, 50, 50, BATCH_SIZE)

# test Encoder
sample_hidden = encoder.initialize_hidden_state()
sample_output, sample_h = encoder(input_dataset, sample_hidden)
print ('Encoder output shape: (batch size, sequence length, units) {}'.format(sample_output.shape))
# print ('Encoder Hidden state shape: (batch size, units) {}'.format(sample_hidden.shape))

Encoder output shape: (batch size, sequence length, units) (32, 82, 100)


In [ ]:
#@title Decoder
import tensorflow_addons as tfa

class Decoder(keras.Model):
  def __init__(self, vocab_size, em_units, dec_units, max_length, batch_size):
    super(Decoder, self).__init__()
    self.dec_units = dec_units
    self.em_units = em_units
    self.vocab_size = vocab_size
    self.max_length = max_length
    self.batch_size = batch_size

    self.embedding = Embedding(vocab_size, em_units)

    self.rnn_cell = LSTMCell(dec_units)
    self.attention = tfa.seq2seq.BahdanauAttention(tf.constant(self.dec_units), probability_fn='softmax')
    self.wrapper = tfa.seq2seq.AttentionWrapper(self.rnn_cell, self.attention, self.dec_units)

    self.sampler = tfa.seq2seq.sampler.TrainingSampler()

    self.fc = Dense(vocab_size, activation='softmax')

    self.decoder = tfa.seq2seq.BasicDecoder(self.wrapper, sampler=self.sampler, output_layer=self.fc)

  def build_initial_state(self, batch_sz, encoder_state, Dtype):
    decoder_initial_state = self.wrapper.get_initial_state(batch_size=batch_sz, dtype=Dtype)
    decoder_initial_state = decoder_initial_state.clone(cell_state=encoder_state)
    return decoder_initial_state

  def call(self, x, initial_state):
    x = self.embedding(x)
    output, _, _ = self.decoder(x, initial_state=initial_state, sequence_length=self.batch_size*[self.max_length-1])

    return output

In [ ]:
#@title decoder test
vocab_size = len(target_tok.word_index)+1
decoder = Decoder(vocab_size, 50, 50, target_dataset.shape[1],BATCH_SIZE)
decoder.attention.setup_memory(sample_output)
initial_state = decoder.build_initial_state(BATCH_SIZE, sample_h, tf.float32)

output = decoder(target_dataset, sample_hidden)

output.shape